# Learning Rate Optimization

Fresh Python 3.11+ CPU kernel. Downloads official MNIST: all 60,000 training records and all 10,000 held-out records. Dataset and reproducible setup supply missing source variables. Retains the original 784→64→10 architectures, batch size 32, 50-epoch step-decay Adam, 10-epoch Adam and 10-epoch Adagrad. Full datasets, no reduced epoch counts. TensorFlow dataset uses a single private worker for constrained hosts. Original examples preserved in appendix.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "tensorflow==2.20.0", "keras==3.15.1", "numpy==2.4.6", "certifi==2026.10.23"], check=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Sequential, Input
from tensorflow.keras.layers import Conv2D, LeakyReLU, Dropout, ZeroPadding2D, BatchNormalization, Flatten, Dense, Reshape, UpSampling2D
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
keras.utils.set_random_seed(42)
np.set_printoptions(precision=6, suppress=True)
print("Python:", sys.version.split()[0], "| TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "torch==2.10.0"], check=True)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.set_num_threads(1)
torch.manual_seed(42)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
train_data = x_train.reshape(-1, 784).astype("float32") / 255.0
test_data = x_test.reshape(-1, 784).astype("float32") / 255.0
train_labels = keras.utils.to_categorical(y_train, 10)
input_dim = 784
options = tf.data.Options()
options.threading.private_threadpool_size = 1
options.threading.max_intra_op_parallelism = 1
train_ds = tf.data.Dataset.from_tensor_slices((train_data, train_labels)).shuffle(60000, seed=42).batch(32).with_options(options)
train_loader = DataLoader(TensorDataset(torch.from_numpy(train_data), torch.from_numpy(y_train.astype("int64"))), batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(42), num_workers=0)
def build_classifier():
    return Sequential([Input(shape=(input_dim,)), Dense(64, activation="relu"), Dense(10, activation="softmax")])
class EpochLog(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: loss={logs['loss']:.6f}, accuracy={logs['accuracy']:.6f}, lr={float(self.model.optimizer.learning_rate.numpy()):.8f}")
print("PyTorch:", torch.__version__)
print("Training:", train_data.shape, "| held out:", test_data.shape, "| batches/epoch:", len(train_loader))

## Example 0

In [ ]:
initial_lr = 0.01
def lr_schedule(epoch):
    return initial_lr * 0.1 ** (epoch // 10)
decay_model = build_classifier()
decay_model.compile(optimizer=keras.optimizers.Adam(learning_rate=initial_lr), loss="categorical_crossentropy", metrics=["accuracy"])
decay_history = decay_model.fit(train_ds, epochs=50, callbacks=[keras.callbacks.LearningRateScheduler(lr_schedule), EpochLog()], verbose=0, shuffle=False)
print("Completed epochs:", len(decay_history.epoch), "| optimizer updates:", int(decay_model.optimizer.iterations.numpy()))

## Example 1

In [ ]:
adam_model = build_classifier()
adam_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="categorical_crossentropy", metrics=["accuracy"])
adam_history = adam_model.fit(train_ds, epochs=10, callbacks=[EpochLog()], verbose=0, shuffle=False)
print("Completed epochs:", len(adam_history.epoch), "| optimizer updates:", int(adam_model.optimizer.iterations.numpy()))

## Example 2

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 10)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))
adagrad_model = SimpleModel()
optimizer = torch.optim.Adagrad(adagrad_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
adagrad_losses = []
for epoch in range(10):
    total_loss, correct, seen = 0.0, 0, 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = adagrad_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach()) * len(labels)
        correct += int((outputs.argmax(1) == labels).sum())
        seen += len(labels)
    assert seen == 60000
    adagrad_losses.append(total_loss / seen)
    print(f"Epoch {epoch+1}: loss={total_loss/seen:.6f}, accuracy={correct/seen:.6f}")

## Independent regression checks

In [ ]:
assert len(train_data) == 60000 and len(test_data) == 10000 and len(train_loader) == 1875
assert len(decay_history.epoch) == 50 and len(adam_history.epoch) == 10 and len(adagrad_losses) == 10
np.testing.assert_allclose(decay_history.history["learning_rate"], [lr_schedule(i) for i in range(50)], rtol=1e-6)
assert int(decay_model.optimizer.iterations.numpy()) == 93750
assert int(adam_model.optimizer.iterations.numpy()) == 18750
assert all(int(optimizer.state[p]["step"]) == 18750 for p in adagrad_model.parameters())
np.testing.assert_allclose(train_labels.sum(axis=1), 1)
for name, model in [("Step decay", decay_model), ("Adam", adam_model)]:
    probabilities = np.concatenate([model(test_data[i:i+256], training=False).numpy() for i in range(0, 10000, 256)])
    assert np.isfinite(probabilities).all()
    np.testing.assert_allclose(probabilities.sum(1), 1, atol=1e-6)
    independent_loss = -np.log(np.maximum(probabilities[np.arange(10000), y_test], 1e-7)).mean()
    independent_accuracy = (probabilities.argmax(1) == y_test).mean()
    expected_loss = keras.losses.CategoricalCrossentropy()(keras.utils.to_categorical(y_test, 10), probabilities).numpy()
    np.testing.assert_allclose(independent_loss, expected_loss, rtol=1e-5)
    print(f"{name} held-out loss={independent_loss:.6f}, accuracy={independent_accuracy:.6f}")
adagrad_model.eval()
with torch.no_grad():
    logits = adagrad_model(torch.from_numpy(test_data))
    reference_loss = criterion(logits, torch.from_numpy(y_test.astype("int64"))).item()
z = logits.numpy().astype("float64")
logsumexp = np.log(np.exp(z-z.max(1,keepdims=True)).sum(1)) + z.max(1)
independent_loss = (logsumexp-z[np.arange(10000), y_test]).mean()
np.testing.assert_allclose(independent_loss, reference_loss, rtol=1e-5)
assert all(torch.isfinite(p).all() and torch.isfinite(optimizer.state[p]["sum"]).all() for p in adagrad_model.parameters())
print(f"Adagrad held-out loss={independent_loss:.6f}, accuracy={(z.argmax(1)==y_test).mean():.6f}")
print("All 70 full-data epochs, 131250 optimizer updates, all 50 scheduled rates and independent held-out loss checks passed.")

## Original source appendix

Full original blocks from raw source SHA-256 e5e677859795ddc9e750652dcf3f4eeeda584a13b0a32b6b398e69219aab5ac8. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import LearningRateScheduler
import numpy as np

# Define a simple neural network model
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    layers.Dense(10, activation='softmax')
])

# Compile the model with an initial learning rate
initial_lr = 0.01
optimizer = optimizers.Adam(learning_rate=initial_lr)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Define a learning rate schedule (step decay)
def lr_schedule(epoch):
    lr = initial_lr * 0.1 ** (epoch // 10)
    return lr

# Set up a learning rate scheduler callback
lr_scheduler = LearningRateScheduler(lr_schedule)

# Train the model with the learning rate scheduler callback
model.fit(train_data, train_labels, epochs=50, batch_size=32, callbacks=[lr_scheduler])
```

### Original block 2

```text
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

# Define a simple neural network model
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    layers.Dense(10, activation='softmax')
])

# Compile the model with Adam optimizer
optimizer = optimizers.Adam(learning_rate=0.001)  # Initial learning rate
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(train_data, train_labels, epochs=10, batch_size=32)
```

### Original block 3

```text
import torch
import torch.nn as nn
import torch.optim as optim

# Define a simple neural network model
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleModel()

# Use Adagrad optimizer
optimizer = optim.Adagrad(model.parameters(), lr=0.01)  # Initial learning rate

# Define loss function
criterion = nn.CrossEntropyLoss()

# Train the model
for epoch in range(10):
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
```